[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/7-verification.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 7. Verification

`verify_cart` is a separate Python checker. It reads stock and the shippable flag itself. It does not trust the quantity, the ship flag, or a missing citation.

A bad cart asks for 100 jars and cites nothing. A good cart is one jar of Calabrian chili oil to Ohio, with a citation. Fresh labneh with `ship` true is rejected. Then the customer asks the seller to check the good cart, and the agent calls the same checker.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## The checker, before the model

These three calls do not use the API. The findings come from the database.

In [ ]:
import json

import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.verify  # registers verify_cart

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.verify import verify_cart

reset_db()
bad = json.loads(verify_cart({
    "sku": "HJ-CHI", "qty": 100, "ship": True, "destination": "Ohio", "citation": "",
}))
good = json.loads(verify_cart({
    "sku": "HJ-CHI", "qty": 1, "ship": True, "destination": "Ohio",
    "citation": "catalog HJ-CHI stock 14",
}))
fresh = json.loads(verify_cart({
    "sku": "HJ-LAB", "qty": 1, "ship": True, "destination": "Ohio",
    "citation": "catalog HJ-LAB",
}))
print("bad", bad)
print("good", good)
print("fresh", fresh)
check(bad["status"] == "rejected", "a quantity over stock with no citation is rejected")
check("qty_exceeds_stock_14" in bad["findings"], "the finding names the stock")
check("missing_citation" in bad["findings"], "a missing citation is a finding")
check(good["status"] == "accepted", "one jar of chili oil to Ohio is accepted")
check(fresh["status"] == "rejected" and "not_shippable" in fresh["findings"], "fresh labneh does not ship")

## The agent calls the checker

The customer asks for the cart that the checker just accepted.

In [ ]:
reset_db()
result = run_agent(
    "Please check a cart with 1 jar of Calabrian chili oil, sku HJ-CHI, shipped to Ohio. Cite the catalog.",
    system_text(),
)
print("ANSWER:", result["text"])
log = "\n".join(result["tool_log"])
check("verify_cart" in log, "the agent called the checker")
check("accepted" in log, "the checker accepted the cart")